# view_http.ipynb — evidence‑first, read‑only

**Purpose**: Inspect *actual* `http.csv` for the current `RELEASE` (from `release.txt`) and mine Scenario‑2 answer keys for job‑search hosts. No writes. No side effects.

**You will use this notebook to:**
- verify raw HTTP schema and basic cardinalities
- cheaply profile URL length/depth shapes
- extract job‑site hosts directly from `answers/<release>-2*` (ground truth)
- optionally check counts of those hosts across the full HTTP table (commented, slow)

**Do not create flags here.** ETL/emit scripts do that. This notebook is the scout, not the factory.

In [ ]:
# Cell 0 — robust import of notebooks/nb_paths no matter where this runs
from pathlib import Path
import sys, importlib.util

def _load_nb_paths():
    # Walk up from CWD until we find notebooks/nb_paths.py
    for cand in [Path.cwd(), *Path.cwd().parents]:
        p = cand / "notebooks" / "nb_paths.py"
        if p.exists():
            spec = importlib.util.spec_from_file_location("notebooks.nb_paths", p)
            mod = importlib.util.module_from_spec(spec)
            sys.modules["notebooks.nb_paths"] = mod
            spec.loader.exec_module(mod)  # type: ignore
            return mod
    raise ModuleNotFoundError("Could not locate notebooks/nb_paths.py from this notebook's CWD.")

_nb = _load_nb_paths()
bootstrap       = _nb.bootstrap
csv_path        = _nb.csv_path
read_csv        = _nb.read_csv
iter_csv_chunks = _nb.iter_csv_chunks

env = bootstrap()
print("Release:", env.RELEASE)

In [ ]:
# Cell 1 — locate raw http.csv and peek a small sample
import pandas as pd
from pathlib import Path

http_csv = csv_path(env, "http")
print("http.csv:", http_csv, "| exists:", http_csv.exists())

if http_csv.exists():
    # tiny peek to avoid blowing RAM
    head = pd.read_csv(http_csv, nrows=20, dtype=str, encoding="utf-8", on_bad_lines="warn")
    display(head)
else:
    print("Missing http.csv. Check release.txt and data folder.")

In [ ]:
# Cell 2 — column inventory + cheap cardinality signals (streaming, read‑only)
import math
from collections import defaultdict

null_counts = defaultdict(int)
row_count = 0
distinct_samples = defaultdict(set)

if http_csv.exists():
    for chunk in iter_csv_chunks(env, "http", chunksize=200_000, dtype=str):
        row_count += len(chunk)
        for c in chunk.columns:
            s = chunk[c].astype(str)
            null_counts[c] += int(s.isna().sum() + (s == 'None').sum())
            # bounded distinct sample to keep memory sane
            if len(distinct_samples[c]) < 2000:
                distinct_samples[c].update(s.dropna().head(500).tolist())
    summary = (
        pd.DataFrame({
            "column": list(null_counts.keys()),
            "null_rate_est": [null_counts[k]/row_count if row_count else 0 for k in null_counts.keys()],
            "approx_distinct_<=2k": [len(distinct_samples[k]) for k in null_counts.keys()],
        }).sort_values(["null_rate_est","column"], ascending=[False, True])
    )
    display(summary)
else:
    print("Missing http.csv.")

In [ ]:
# Cell 3 — light URL profiling: length & depth bins via DuckDB (fast, no Python urlparse)
import duckdb
p = str(http_csv)

if Path(p).exists():
    base_sql = f"""
    WITH src AS (
      SELECT url
      FROM read_csv_auto('{p}', columns={{'url':'VARCHAR'}}, header=TRUE)
    ),
    norm AS (
      SELECT
        lower(regexp_extract(url, '^(?:https?://)?([^/?:#]+)', 1)) AS host,
        length(url) AS url_len,
        COALESCE(length(regexp_replace(COALESCE(regexp_extract(url, '^(?:https?://)?[^/?:#]+(.*)$', 1), ''), '[^/]', '', 'g')), 0) AS url_depth
      FROM src WHERE url IS NOT NULL
    )
    """

    con = duckdb.connect(database=":memory:")
    len_bins = con.execute(base_sql + """
      SELECT CASE
               WHEN url_len < 30  THEN '<30'
               WHEN url_len < 60  THEN '30-59'
               WHEN url_len < 120 THEN '60-119'
               WHEN url_len < 240 THEN '120-239'
               ELSE '240+'
             END AS url_length_bin,
             COUNT(*) AS rows
      FROM norm GROUP BY 1 ORDER BY 1
    """).fetchdf()
    depth_bins = con.execute(base_sql + """
      SELECT CASE
               WHEN url_depth = 0 THEN '0'
               WHEN url_depth = 1 THEN '1'
               WHEN url_depth = 2 THEN '2'
               WHEN url_depth = 3 THEN '3'
               ELSE '4+'
             END AS url_depth_bin,
             COUNT(*) AS rows
      FROM norm GROUP BY 1 ORDER BY 1
    """).fetchdf()
    display(len_bins); display(depth_bins)
else:
    print("Missing http.csv.")

## Scenario 2 ground‑truth job‑site discovery
Extract hosts from `answers/<release>-2*` (unlabeled CSVs). Use this list to seed ETL job‑site flags. No guesses.

In [ ]:
# Cell 4 — derive job-site hosts from Scenario 2 answer keys
from urllib.parse import urlparse
import re, pandas as pd

answers_dir = env.PROJECT / "answers"
sc2_files = sorted(answers_dir.glob(f"{env.RELEASE}-2*"))
print("Scenario 2 files:", [p.name for p in sc2_files])
if not sc2_files:
    raise FileNotFoundError(f"No Scenario 2 answers found for {env.RELEASE}")

def read_unlabeled(p: Path) -> pd.DataFrame:
    return pd.read_csv(p, header=None, dtype=str, encoding="utf-8", on_bad_lines="skip")

url_re = re.compile(r"https?://\S+", re.I)
urls = []
for p in sc2_files:
    df = read_unlabeled(p)
    vals = df.stack(dropna=True).astype(str)
    urls.extend([m.group(0) for s in vals for m in url_re.finditer(s)])

def host_of(u: str):
    try:
        h = urlparse(u).hostname
        return h.lower() if h else None
    except Exception:
        return None

def registrable(host: str):
    parts = (host or '').split('.')
    return '.'.join(parts[-2:]) if len(parts) >= 2 else host

hosts = pd.Series([registrable(host_of(u)) for u in urls], dtype="string").dropna()
jobsite_candidates = hosts.value_counts().rename_axis("host").to_frame("count").sort_values("count", ascending=False)
print("Scenario-2 answer hosts (ranked):")
display(jobsite_candidates.head(100))

# Handy set you can paste into ETL after review
jobsite_set = set(jobsite_candidates.head(50).index.tolist())
print("jobsite_set =", jobsite_set)

In [ ]:
# Cell 5 — OPTIONAL and SLOW (commented): count those hosts across full HTTP
# This sweeps the entire http.csv to count occurrences of the discovered hosts.
# Keep it commented; results won't change unless the data changes.
"""
from urllib.parse import urlparse
import pandas as pd

target = set(jobsite_candidates.head(50).index)
hits = {k: 0 for k in target}

for chunk in iter_csv_chunks(env, "http", chunksize=200_000, dtype=str):
    if "url" not in chunk.columns:
        continue
    hosts = chunk["url"].dropna().astype(str).map(lambda u: (urlparse(u).hostname or "").lower())
    for h, ct in hosts.value_counts().items():
        if h in hits:
            hits[h] += int(ct)

dom_df = pd.DataFrame([(k, v) for k, v in hits.items() if v > 0], columns=["host","http_hits"]).sort_values("http_hits", ascending=False)
display(dom_df)
"""

## HTTP ETL quick actions

Use these two cells. Run them from the repo root (or `cd` first).

### 1) Rebuild HTTP parquets (idempotent)
Writes `out/<release>/http_v3/http_v3_full/http_full.parquet` and `http_v3_lean/http_lean.parquet`.
```python
# rebuild http (uses lean LDAP + LOGON helpers)
!python -m scripts.etl http \
  --profile lean \
  --family http_v3 \
  --ldap-family-for-join ldap_v3_lean \
  --logon-family-for-pc logon_v3_lean \
  --overwrite
```

### 2) Emit HTTP QC sidecars only (no rebuild)
Refreshes `out/<release>/qc/http_meta.json` and `http_qc.md` from the parquets already on disk.
```python
# write QC sidecars without touching the parquets
python - <<'PY'
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
from notebooks.nb_paths import bootstrap
from src.helpers.emit import emit_http_final
env = bootstrap()
emit_http_final(env, df_full=None, df_lean=None, family="http_v3", overwrite=False)
print("http QC sidecars written.")
PY

### Notes
- Flags like `is_job_site`, `is_wikileaks`, `is_dropbox`, `url_len`, `url_depth` should be implemented in ETL, not here.
- This notebook should never write to `out/`.
- If a future release adds columns (e.g., `activity`), Cells 1–2 will show it and you can adapt ETL safely.